# Ders 1: Zaman Serisi Analizine Giriş
- 1.1 Zaman Serisi Örnekleri
- 1.2 Zaman Serisi Analizinin Amaçları
- 1.3 Basit Zaman Serisi Modelleri
- 1.4 Durağan Modeller ve Otokorelasyon Fonksiyonu
- 1.5 Trend ve Mevsimsellik Tahmini ve Giderilmesi
- 1.6 Gürültü Dizisinin Test Edilmesi


In [ ]:
# Gerekli kütüphaneleri yükleyelim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12
print("Kütüphaneler başarıyla yüklendi!")


## 1.1 Zaman Serisi Örnekleri

Bir **zaman serisi**, belirli zaman noktalarında kaydedilen gözlemler kümesidir:
$$\{x_t, t \in T_0\}$$

Kitaptaki temel örnekleri Python ile yeniden oluşturalım.


In [ ]:
# ── Örnek 1: Avustralya Kırmızı Şarap Satışları (simüle edilmiş) ──
np.random.seed(42)
n = 142  # Oca 1980 – Eki 1991
t = np.arange(n)

# Trend + mevsimsellik + gürültü
trend = 0.01 * t
seasonal = 0.3 * np.sin(2 * np.pi * t / 12) + 0.1 * np.cos(2 * np.pi * t / 12)
noise = np.random.normal(0, 0.08, n)
wine = 1.0 + trend + seasonal + noise

dates = pd.date_range('1980-01', periods=n, freq='ME')

fig, axes = plt.subplots(3, 1, figsize=(13, 10))

# Şarap satışları
axes[0].plot(dates, wine, color='darkred', lw=1.5)
axes[0].set_title('Avustralya Kırmızı Şarap Satışları (Oca 1980 – Eki 1991)', fontsize=13)
axes[0].set_ylabel('Satışlar (bin kL)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Kazalı ölümler (ABD, 1973-1978)
n2 = 72
t2 = np.arange(n2)
deaths = (9 + 0.5*np.sin(2*np.pi*t2/12) + np.random.normal(0, 0.3, n2))
dates2 = pd.date_range('1973-01', periods=n2, freq='ME')
axes[1].plot(dates2, deaths, color='steelblue', lw=1.5)
axes[1].set_title('ABD Kazalı Ölümleri (1973–1978)', fontsize=13)
axes[1].set_ylabel('Ölümler (bin)')

# ABD Nüfusu (1790-1990)
years = np.arange(1790, 2000, 10)
pop = 3.9 * np.exp(0.02 * (years - 1790)) + np.random.normal(0, 1, len(years))
pop = np.clip(pop, 0, None)
axes[2].plot(years, pop, 'o-', color='green', lw=1.5, markersize=4)
axes[2].set_title('ABD Nüfusu (1790–1990)', fontsize=13)
axes[2].set_ylabel('Nüfus (Milyon)')

plt.tight_layout()
plt.show()
print("Üç farklı zaman serisi türü görüldü:")
print("  1. Trend + Mevsimsellik (şarap)")
print("  2. Mevsimsellik (ölümler)")
print("  3. Üstel trend (nüfus)")


## 1.3 Basit Zaman Serisi Modelleri

### 1.3.1 Sıfır-Ortalıklı Modeller

**Beyaz Gürültü (White Noise):** $\{Z_t\} \sim WN(0, \sigma^2)$, yani $E[Z_t]=0$, $\text{Cov}(Z_t, Z_s)=0$ ($t\neq s$).

**Hareketli Ortalama (MA):** $X_t = \frac{1}{3}(Z_{t-1} + Z_t + Z_{t+1})$

**Otoregresif (AR):** $X_t = \phi X_{t-1} + Z_t$


In [ ]:
# ── Sıfır Ortalıklı Modeller ──
np.random.seed(0)
n = 200
Z = np.random.normal(0, 1, n + 2)

# Beyaz Gürültü
WN = Z[1:-1]

# Hareketli Ortalama MA(1): 3-nokta
MA = np.array([(Z[t-1] + Z[t] + Z[t+1]) / 3 for t in range(1, n+1)])

# AR(1): X_t = 0.9 * X_{t-1} + Z_t
AR = np.zeros(n)
AR[0] = Z[1]
phi = 0.9
for t in range(1, n):
    AR[t] = phi * AR[t-1] + Z[t+1]

fig, axes = plt.subplots(3, 1, figsize=(13, 9))
models = [('Beyaz Gürültü $Z_t$', WN, 'gray'),
          ('Hareketli Ortalama $X_t = (Z_{t-1}+Z_t+Z_{t+1})/3$', MA, 'steelblue'),
          (r'AR(1): $X_t = 0.9 X_{t-1} + Z_t$', AR, 'darkorange')]

for ax, (title, data, color) in zip(axes, models):
    ax.plot(data, color=color, lw=1.2)
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Zaman')

plt.tight_layout()
plt.show()

print(f"Beyaz Gürültü  — Ort: {WN.mean():.3f}, Std: {WN.std():.3f}")
print(f"Hareketli Ort. — Ort: {MA.mean():.3f}, Std: {MA.std():.3f}")
print(f"AR(1) φ=0.9   — Ort: {AR.mean():.3f}, Std: {AR.std():.3f}")


### 1.3.2 Trend ve Mevsimselliği Olan Modeller

Genel model:
$$X_t = m_t + s_t + Y_t$$

Burada:
- $m_t$: deterministik trend bileşeni
- $s_t$: mevsimsel bileşen ($\sum_{j=1}^{d} s_j = 0$)
- $Y_t$: durağan gürültü süreci


In [ ]:
# ── Trend + Mevsimsellik Ayrıştırma ──
np.random.seed(7)
n = 96  # 8 yıl, aylık veri
t = np.arange(1, n + 1)

# Bileşenler
trend = 0.05 * t          # Doğrusal trend
seasonal = 2 * np.sin(2 * np.pi * t / 12)   # Aylık mevsimsellik
noise = np.random.normal(0, 0.5, n)
X = trend + seasonal + noise

fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)
axes[0].plot(t, X, 'k', lw=1.2); axes[0].set_title('Gözlemlenen Seri $X_t$')
axes[1].plot(t, trend, 'r', lw=1.5); axes[1].set_title('Trend $m_t = 0.05t$')
axes[2].plot(t, seasonal, 'b', lw=1.5); axes[2].set_title('Mevsimsellik $s_t$')
axes[3].plot(t, noise, 'g', lw=1.2); axes[3].set_title('Gürültü $Y_t$')
for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.set_ylabel('Değer')
axes[-1].set_xlabel('Zaman (Ay)')
plt.tight_layout()
plt.show()


## 1.4 Durağan Modeller ve Otokorelasyon Fonksiyonu

Bir $\{X_t\}$ süreci **(zayıf) durağan**dır eğer:
1. $E[X_t] = \mu$ (sabit ortalama)
2. $\gamma(t+h, t) = \text{Cov}(X_{t+h}, X_t)$ yalnızca $h$'ye bağlıdır

**Otokovaryans fonksiyonu:** $\gamma(h) = \text{Cov}(X_{t+h}, X_t)$

**Otokorelasyon fonksiyonu (ACF):** $\rho(h) = \frac{\gamma(h)}{\gamma(0)}$

**Örneklem ACF:**
$$\hat{\rho}(h) = \frac{\sum_{t=1}^{n-h}(X_{t+h} - \bar{X})(X_t - \bar{X})}{\sum_{t=1}^{n}(X_t - \bar{X})^2}$$


In [ ]:
# ── ACF Hesaplama ve Yorumlama ──
np.random.seed(42)
n = 300

# Farklı süreçler
WN = np.random.normal(0, 1, n)

# AR(1) phi=0.8
AR1_pos = np.zeros(n)
AR1_pos[0] = WN[0]
for t in range(1, n):
    AR1_pos[t] = 0.8 * AR1_pos[t-1] + np.random.normal(0, 1)

# AR(1) phi=-0.8
AR1_neg = np.zeros(n)
AR1_neg[0] = WN[0]
for t in range(1, n):
    AR1_neg[t] = -0.8 * AR1_neg[t-1] + np.random.normal(0, 1)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
series = [('Beyaz Gürültü', WN), ('AR(1) φ=+0.8', AR1_pos), ('AR(1) φ=-0.8', AR1_neg)]

for i, (name, s) in enumerate(series):
    axes[i, 0].plot(s[:100], lw=1, color='steelblue')
    axes[i, 0].set_title(f'{name} — Zaman Serisi')
    axes[i, 0].axhline(0, color='gray', ls='--', lw=0.5)
    
    plot_acf(s, lags=20, ax=axes[i, 1], color='steelblue', title=f'{name} — ACF')

plt.tight_layout()
plt.show()

print("Teori doğrulaması:")
print(f"AR(1) φ=0.8  → Teorik ρ(1) = {0.8:.3f},  Hesaplanan ρ(1) ≈ {acf(AR1_pos, nlags=1)[1]:.3f}")
print(f"AR(1) φ=-0.8 → Teorik ρ(1) = {-0.8:.3f}, Hesaplanan ρ(1) ≈ {acf(AR1_neg, nlags=1)[1]:.3f}")


## 1.5 Trend ve Mevsimselliğin Tahmini ve Giderilmesi

### 1.5.1 Mevsimsellik Yokken Trend Tahmini

**Hareketli Ortalama (Moving Average) Filtresi:**
$$\hat{m}_t = \frac{1}{2q+1}\sum_{j=-q}^{q} X_{t+j}$$

**Polinom Regresyonu:** $m_t = a_0 + a_1 t + a_2 t^2 + \ldots$


In [ ]:
# ── Trend Tahmini: Hareketli Ortalama vs Polinom Regresyon ──
np.random.seed(10)
n = 100
t = np.arange(1, n+1)

# Gerçek trend: kuadratik
true_trend = 0.002 * t**2 - 0.1 * t + 5
X = true_trend + np.random.normal(0, 0.8, n)

# 1) Hareketli Ortalama (q=5, pencere=11)
q = 5
MA_trend = np.full(n, np.nan)
for i in range(q, n-q):
    MA_trend[i] = X[i-q:i+q+1].mean()

# 2) Polinom Regresyon (derece=2)
coeffs = np.polyfit(t, X, 2)
poly_trend = np.polyval(coeffs, t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t, X, 'lightgray', lw=1, label='Gözlem')
axes[0].plot(t, true_trend, 'k--', lw=2, label='Gerçek Trend')
axes[0].plot(t, MA_trend, 'r', lw=2, label=f'MA (q={q})')
axes[0].plot(t, poly_trend, 'b', lw=2, label='Polinom (derece 2)')
axes[0].legend(); axes[0].set_title('Trend Tahmin Yöntemleri')

# Artıklar
resid_poly = X - poly_trend
axes[1].plot(t, resid_poly, color='steelblue', lw=1)
axes[1].axhline(0, color='red', ls='--')
axes[1].set_title('Trend Çıkarıldıktan Sonra Artıklar')
axes[1].set_xlabel('Zaman')

plt.tight_layout()
plt.show()

print(f"Polinom katsayıları: a2={coeffs[0]:.4f}, a1={coeffs[1]:.4f}, a0={coeffs[2]:.4f}")
print(f"Gerçek katsayılar :  a2=0.0020,      a1=-0.1000,     a0=5.0000")


### 1.5.2 Hem Trend Hem Mevsimsellik Tahmin ve Giderme

Önce mevsimsel farklama uygulayarak trend çıkarılır:
$$\nabla_d X_t = X_t - X_{t-d}$$

Ardından trend için ikinci bir farklama:
$$\nabla \nabla_d X_t = (1-B)(1-B^d)X_t$$


In [ ]:
# ── Trend + Mevsimsellik Ayrıştırma: Klasik Yöntem ──
from statsmodels.tsa.seasonal import seasonal_decompose

np.random.seed(5)
n = 120  # 10 yıl, aylık
t_idx = np.arange(1, n+1)

trend_comp = 0.05 * t_idx
seas_comp = 1.5 * np.sin(2 * np.pi * t_idx / 12) + 0.3 * np.sin(4 * np.pi * t_idx / 12)
noise_comp = np.random.normal(0, 0.25, n)
X = trend_comp + seas_comp + noise_comp

dates = pd.date_range('2013-01', periods=n, freq='ME')
ts = pd.Series(X, index=dates)

# Mevsimsel ayrıştırma
result = seasonal_decompose(ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
result.observed.plot(ax=axes[0], color='k', lw=1.2, title='Gözlemlenen')
result.trend.plot(ax=axes[1], color='r', lw=1.5, title='Trend')
result.seasonal.plot(ax=axes[2], color='b', lw=1.5, title='Mevsimsellik')
result.resid.plot(ax=axes[3], color='g', lw=1.2, title='Artık (Residual)')
for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls='--')
plt.tight_layout()
plt.show()

# Farklama ile mevsimsellik giderme
diff12 = ts.diff(12).dropna()
diff1_12 = diff12.diff(1).dropna()

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=False)
ts.plot(ax=axes[0], title='Orijinal Seri $X_t$', color='k')
diff12.plot(ax=axes[1], title='Mevsimsel Farklama $\\nabla_{12} X_t$', color='blue')
diff1_12.plot(ax=axes[2], title='Mevsimsel + Trend Farklama $\\nabla \\nabla_{12} X_t$', color='green')
plt.tight_layout()
plt.show()


## 1.6 Gürültü Dizisinin Test Edilmesi

Artıkların bağımsızlığını test etmek için:
- **Portmanteau (Ljung-Box) testi:** $H_0$: İlk $h$ lag'da otokorelasyon yoktur
$$Q = n(n+2) \sum_{j=1}^{h} \frac{\hat{\rho}^2(j)}{n-j} \sim \chi^2(h)$$


In [ ]:
# ── Ljung-Box Testi ──
from statsmodels.stats.diagnostic import acorr_ljungbox

np.random.seed(99)

# Test 1: Gerçek Beyaz Gürültü
WN_test = np.random.normal(0, 1, 200)

# Test 2: AR(1) artıkları (yanlış model)
AR_test = np.zeros(200)
for t in range(1, 200):
    AR_test[t] = 0.7 * AR_test[t-1] + np.random.normal(0, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (name, s, color) in enumerate([('Beyaz Gürültü', WN_test, 'steelblue'), 
                                        ('AR(1) Artıkları', AR_test, 'crimson')]):
    axes[i, 0].plot(s, lw=0.8, color=color)
    axes[i, 0].axhline(0, color='k', ls='--', lw=0.5)
    axes[i, 0].set_title(f'{name} — Zaman Serisi')
    plot_acf(s, lags=20, ax=axes[i, 1], color=color, title=f'{name} — ACF')

plt.tight_layout()
plt.show()

print("=" * 55)
print(f"{'Seri':<20} {'Lag':>5} {'LB İstatistiği':>15} {'p-değeri':>12}")
print("=" * 55)
for name, s in [('Beyaz Gürültü', WN_test), ('AR(1) Artıkları', AR_test)]:
    lb = acorr_ljungbox(s, lags=[10, 20], return_df=True)
    for lag, row in lb.iterrows():
        print(f"{name:<20} {int(lag):>5} {row['lb_stat']:>15.3f} {row['lb_pvalue']:>12.4f}")
    print()

print("p < 0.05 → H0 reddedilir → Artıklar bağımsız DEĞİL (model yetersiz)")
print("p > 0.05 → H0 reddedilemez → Artıklar bağımsız görünüyor (iyi model)")


## Özet

| Kavram | Açıklama |
|--------|----------|
| **Zaman Serisi** | Zamana bağlı gözlem dizisi |
| **Trend** | Uzun dönem yönelim ($m_t$) |
| **Mevsimsellik** | Periyodik tekrarlayan örüntü ($s_t$) |
| **Durağanlık** | Sabit ortalama ve kovaryans yapısı |
| **ACF** | Gecikmeli korelasyon ölçüsü $\rho(h)$ |
| **Ljung-Box** | Artıkların bağımsızlık testi |

